# RAG at Scale: Multi-Tenant Isolation — Runnable Notebook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/anacodicAI-labs/learn-notebooks/blob/main/genai-design/9-5/rag-at-scale.ipynb) &nbsp; [View on GitHub](https://github.com/anacodicAI-labs/learn-notebooks/blob/main/genai-design/9-5/rag-at-scale.ipynb)

Companion to: [RAG at Scale](https://anacodicai.org/learn/genai-design/9-5/rag-at-scale)


## What you'll build

1. A tiny shared vector index holding two tenants' documents.
2. An **unrestricted** search that can leak a different tenant's document.
3. A **namespace-restricted** search that provably cannot — enforced by an
   assertion, not just observed.

**No API key needed.** Everything below is pure Python standard library.

### Setup

In [1]:
import re
import math
from collections import Counter
print("ready — standard library only")

ready — standard library only


## Step 1 — two tenants, one shared index

Both tenants happen to write similar kinds of support documentation — exactly
the situation where cross-tenant leakage is both likely and dangerous, because
a wrong-tenant result still *reads* as plausible.

In [2]:
DOCS = {
    ("tenant_a", "a1"): "Reset your device by holding the power button for "
        "ten seconds until the status light blinks twice.",
    ("tenant_a", "a2"): "Billing cycles run monthly and invoices are emailed "
        "on the first business day after the cycle closes.",
    ("tenant_a", "a3"): "Escalate a support ticket to tier two if it remains "
        "unresolved after twenty four hours.",
    ("tenant_b", "b1"): "Reset the unit by pressing the reset pinhole for "
        "five seconds until the indicator turns solid blue.",
    ("tenant_b", "b2"): "Billing disputes must be filed within thirty days "
        "of the statement date to qualify for a refund.",
    ("tenant_b", "b3"): "Priority accounts get a two hour response window "
        "instead of the standard next-business-day window.",
}
print(f"{len(DOCS)} documents across 2 tenants, one shared index")

6 documents across 2 tenants, one shared index


## Step 2 — a simple stemmed similarity search

Nothing fancy — word-frequency cosine similarity with light suffix-stripping,
so "billed" and "billing" match each other.

In [3]:
def stem(word):
    for suffix in ("ing", "ed", "s"):
        if word.endswith(suffix) and len(word) - len(suffix) >= 3:
            return word[: -len(suffix)]
    return word

def vectorize(text):
    return Counter(stem(w) for w in re.findall(r"[a-z]+", text.lower()))

def cosine(a, b):
    common = set(a) & set(b)
    dot = sum(a[w] * b[w] for w in common)
    mag_a = math.sqrt(sum(v * v for v in a.values()))
    mag_b = math.sqrt(sum(v * v for v in b.values()))
    return dot / (mag_a * mag_b) if mag_a and mag_b else 0.0

INDEX = {key: vectorize(text) for key, text in DOCS.items()}

def search(query, k=1, restrict_tenant=None):
    q_vec = vectorize(query)
    candidates = [
        (key, cosine(q_vec, vec))
        for key, vec in INDEX.items()
        if restrict_tenant is None or key[0] == restrict_tenant
    ]
    candidates.sort(key=lambda kv: -kv[1])
    return candidates[:k]

print("search() ready — restrict_tenant=None means UNRESTRICTED")

search() ready — restrict_tenant=None means UNRESTRICTED


## Step 3 — run both modes side by side

**Try it:** add your own query below (as `(asking_tenant, "your question")`)
and see whether it leaks under the unrestricted mode.

In [4]:
QUERIES = [
    ("tenant_a", "how do I reset my device"),
    ("tenant_a", "when will I be billed"),
    ("tenant_b", "how do I reset my unit"),
    ("tenant_b", "how long to dispute a charge"),
]

print(f"{'asking tenant':14}{'query':32}{'unrestricted top-1':22}{'restricted top-1':22}")
leak_count = 0
for asking_tenant, query in QUERIES:
    unrestricted = search(query, k=1, restrict_tenant=None)
    restricted = search(query, k=1, restrict_tenant=asking_tenant)
    u_key, r_key = unrestricted[0][0], restricted[0][0]
    leaked = u_key[0] != asking_tenant
    if leaked:
        leak_count += 1
    print(f"{asking_tenant:14}{query:32}{str(u_key):22}{str(r_key):22}"
          f"{'  <-- LEAKED' if leaked else ''}")

print(f"\nunrestricted search: {leak_count}/{len(QUERIES)} queries leaked "
      f"another tenant's document")

asking tenant query                           unrestricted top-1    restricted top-1      
tenant_a      how do I reset my device        ('tenant_a', 'a1')    ('tenant_a', 'a1')    
tenant_a      when will I be billed           ('tenant_b', 'b2')    ('tenant_a', 'a2')      <-- LEAKED
tenant_b      how do I reset my unit          ('tenant_b', 'b1')    ('tenant_b', 'b1')    
tenant_b      how long to dispute a charge    ('tenant_b', 'b2')    ('tenant_b', 'b2')    

unrestricted search: 1/4 queries leaked another tenant's document


## Step 4 — prove it, don't just observe it

The restricted path isn't "usually fine" — it's structurally incapable of
returning another tenant's document, because that document was never even a
candidate. The assertion below runs on every query, not just the ones that
happened to leak above.

In [5]:
for asking_tenant, query in QUERIES:
    restricted = search(query, k=1, restrict_tenant=asking_tenant)
    r_key = restricted[0][0]
    assert r_key[0] == asking_tenant, "restricted search must never cross tenants"

print("all 4 restricted-search assertions passed: 0 leaks, by construction")

all 4 restricted-search assertions passed: 0 leaks, by construction


## Try it yourself

- Add a third tenant with documents that overlap in vocabulary with both
  existing tenants, and see how often unrestricted search leaks now.
- Remove the `stem()` step and re-run — does the unrestricted leak count in
  Step 3 change? (It should get *worse*, since exact-word matching is even
  less discriminating.)
- This is the same underlying lesson as the chunking notebook: a similarity
  score tells you what's *relevant*, never what's *permitted* — those need
  two separate checks.